<a href="https://colab.research.google.com/github/famub/2026-GP1-02/blob/main/models/ResNet101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
!pip install kagglehub
import kagglehub
path = kagglehub.dataset_download("shashwatwork/knee-osteoarthritis-dataset-with-severity")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'knee-osteoarthritis-dataset-with-severity' dataset.
Path to dataset files: /kaggle/input/knee-osteoarthritis-dataset-with-severity


In [ ]:
!ls {path}

auto_test  test  train	val


In [ ]:
from torchvision import transforms, datasets

base = path

tf = transforms.Compose([
    transforms.Grayscale(3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train = datasets.ImageFolder(f"{base}/train", transform=tf)
val   = datasets.ImageFolder(f"{base}/val",   transform=tf)
test  = datasets.ImageFolder(f"{base}/test",  transform=tf)

print("train:", len(train), "| val:", len(val), "| test:", len(test))
print("classes:", train.classes)

train: 5778 | val: 826 | test: 1656
classes: ['0', '1', '2', '3', '4']


In [ ]:
from collections import Counter

for name, ds in [("train", train), ("val", val), ("test", test)]:
    counts = Counter(ds.targets)
    print(name, {ds.classes[k]: v for k, v in sorted(counts.items())})

train {'0': 2286, '1': 1046, '2': 1516, '3': 757, '4': 173}
val {'0': 328, '1': 153, '2': 212, '3': 106, '4': 27}
test {'0': 639, '1': 296, '2': 447, '3': 223, '4': 51}


In [ ]:
import torch

counts = Counter(train.targets)
total = sum(counts.values())
weights = [total / (5 * counts[i]) for i in range(5)]
class_weights = torch.tensor(weights, dtype=torch.float)

print("class weights:", [round(w, 2) for w in weights])

class weights: [0.51, 1.1, 0.76, 1.53, 6.68]


In [ ]:
train_tf = transforms.Compose([
    transforms.Grayscale(3),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train = datasets.ImageFolder(f"{base}/train", transform=train_tf)
print("train:", len(train))

train: 5778


In [ ]:
from torch.utils.data import DataLoader

val  = datasets.ImageFolder(f"{base}/val",  transform=tf)
test = datasets.ImageFolder(f"{base}/test", transform=tf)

train_loader = DataLoader(train, batch_size=32, shuffle=True)
val_loader   = DataLoader(val,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test,  batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
print("batch shape:", images.shape)
print("labels:", labels[:10])

batch shape: torch.Size([32, 3, 224, 224])
labels: tensor([2, 1, 0, 2, 0, 2, 0, 0, 0, 1])


In [ ]:
import cv2
import numpy as np
from PIL import Image

# دالة التنظيف:
class Denoise:
    def __call__(self, img):
        g = np.array(img.convert("L"))
        g = cv2.GaussianBlur(g, (3, 3), 0)
        g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
        return Image.fromarray(g)

# pipeline للـ val و test (بدون augmentation)
tf = transforms.Compose([
    Denoise(),
    transforms.Grayscale(3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# pipeline للـ train (مع augmentation)
train_tf = transforms.Compose([
    Denoise(),
    transforms.Grayscale(3),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train = datasets.ImageFolder(f"{base}/train", transform=train_tf)
val   = datasets.ImageFolder(f"{base}/val",   transform=tf)
test  = datasets.ImageFolder(f"{base}/test",  transform=tf)

train_loader = DataLoader(train, batch_size=32, shuffle=True)
val_loader   = DataLoader(val,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test,  batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
print("batch shape:", images.shape)
print("train:", len(train), "| val:", len(val), "| test:", len(test))

batch shape: torch.Size([32, 3, 224, 224])
train: 5778 | val: 826 | test: 1656


In [ ]:
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.resnet101(weights='IMAGENET1K_V2')

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5)

model = model.to(device)

print("Output features:", model.fc)

Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:01<00:00, 151MB/s]


Output features: Linear(in_features=2048, out_features=5, bias=True)


In [ ]:
import torch.nn as nn
import torch.optim as optim

# 1. تعريف دالة الخسارة مع أخذ أوزان الفئات بعين الاعتبار لمعالجة تباين البيانات
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# 2. تعريف المحسن (Optimizer) لتحديث أوزان الشبكة
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. حلقة التدريب والتحقق (Training and Validation Loop)
num_epochs = 3  # عدد الدورات للتجربة

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 30)

    # --- مرحلة التدريب ---
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    train_loss = running_loss / len(train_loader.dataset)
    train_acc = running_corrects.double() / len(train_loader.dataset)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

    # --- مرحلة التحقق (Validation) ---
    model.eval()
    val_loss = 0.0
    val_corrects = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

    val_loss_epoch = val_loss / len(val_loader.dataset)
    val_acc_epoch = val_corrects.double() / len(val_loader.dataset)
    print(f"Val Loss:   {val_loss_epoch:.4f} | Val Acc:   {val_acc_epoch:.4f}")



Epoch 1/3
------------------------------
Train Loss: 1.2888 | Train Acc: 0.3840
Val Loss:   1.4707 | Val Acc:   0.2663

Epoch 2/3
------------------------------
Train Loss: 1.0578 | Train Acc: 0.4822
Val Loss:   1.1030 | Val Acc:   0.5363

Epoch 3/3
------------------------------
Train Loss: 0.9792 | Train Acc: 0.5201
Val Loss:   1.9736 | Val Acc:   0.3015

تم تدريب نموذج ResNet101 بنجاح تام!


In [ ]:
import torch.nn as nn
import torch.optim as optim
import copy

# 1. إعداد دالة الخسارة والمحسن (Optimizer)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. تحديد عدد الدورات للتدريب الفعلي
num_epochs = 20

# متغيرات لتتبع وحفظ أفضل نموذج
best_model_wts = copy.deepcopy(model.state_dict())
best_val_acc = 0.0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 35)

    # --- مرحلة التدريب ---
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    train_loss = running_loss / len(train_loader.dataset)
    train_acc = running_corrects.double() / len(train_loader.dataset)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

    # --- مرحلة التحقق (Validation) ---
    model.eval()
    val_loss = 0.0
    val_corrects = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

    val_loss_epoch = val_loss / len(val_loader.dataset)
    val_acc_epoch = val_corrects.double() / len(val_loader.dataset)
    print(f"Val Loss:   {val_loss_epoch:.4f} | Val Acc:   {val_acc_epoch:.4f}")

    # تحديث وحفظ أفضل نموذج إذا تحسنت دقة التحقق
    if val_acc_epoch > best_val_acc:
        best_val_acc = val_acc_epoch
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), 'best_resnet101_knee.pth')
        print("تم تحديث وحفظ أفضل نموذج بنجاح.")

print(f"\nانتهى التدريب بنجاح .")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")


Epoch 1/20
-----------------------------------
Train Loss: 0.9828 | Train Acc: 0.5261
Val Loss:   1.2043 | Val Acc:   0.4407
تم تحديث وحفظ أفضل نموذج بنجاح.

Epoch 2/20
-----------------------------------
Train Loss: 0.9036 | Train Acc: 0.5516
Val Loss:   1.0611 | Val Acc:   0.5799
تم تحديث وحفظ أفضل نموذج بنجاح.

Epoch 3/20
-----------------------------------
Train Loss: 0.8408 | Train Acc: 0.5924
Val Loss:   1.1644 | Val Acc:   0.5400

Epoch 4/20
-----------------------------------
Train Loss: 0.8242 | Train Acc: 0.5997
Val Loss:   1.2217 | Val Acc:   0.5182

Epoch 5/20
-----------------------------------
Train Loss: 0.7704 | Train Acc: 0.6194
Val Loss:   0.9446 | Val Acc:   0.5969
تم تحديث وحفظ أفضل نموذج بنجاح.

Epoch 6/20
-----------------------------------
Train Loss: 0.7247 | Train Acc: 0.6295
Val Loss:   1.1422 | Val Acc:   0.5896

Epoch 7/20
-----------------------------------
Train Loss: 0.7736 | Train Acc: 0.6184
Val Loss:   0.9314 | Val Acc:   0.6065
تم تحديث وحفظ أفضل نمو

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy

# 1. إعداد دالة الخسارة والمحسن
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. إعداد جدول تخفيض معدل التعلم بدون استخدام الوسيط المزال
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# 3. تحديد عدد الدورات بـ 30 دورة كاملة
num_epochs = 30

best_model_wts = copy.deepcopy(model.state_dict())
best_val_acc = 0.0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 35)

    # --- مرحلة التدريب ---
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    train_loss = running_loss / len(train_loader.dataset)
    train_acc = running_corrects.double() / len(train_loader.dataset)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

    # --- مرحلة التحقق (Validation) ---
    model.eval()
    val_loss = 0.0
    val_corrects = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

    val_loss_epoch = val_loss / len(val_loader.dataset)
    val_acc_epoch = val_corrects.double() / len(val_loader.dataset)
    print(f"Val Loss:   {val_loss_epoch:.4f} | Val Acc:   {val_acc_epoch:.4f}")

    # تحديث معدل التعلم بناءً على خسارة التحقق
    scheduler.step(val_loss_epoch)

    # حفظ أفضل نموذج
    if val_acc_epoch > best_val_acc:
        best_val_acc = val_acc_epoch
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), 'best_resnet101_knee.pth')
        print("تم تحديث وحفظ أفضل نموذج بنجاح.")

print(f"\nانتهى التدريب بنجاح .")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")


Epoch 1/30
-----------------------------------
Train Loss: 0.5215 | Train Acc: 0.7288
Val Loss:   1.1189 | Val Acc:   0.5763
تم تحديث وحفظ أفضل نموذج بنجاح.

Epoch 2/30
-----------------------------------
Train Loss: 0.4893 | Train Acc: 0.7309
Val Loss:   1.0612 | Val Acc:   0.6199
تم تحديث وحفظ أفضل نموذج بنجاح.

Epoch 3/30
-----------------------------------
Train Loss: 0.4654 | Train Acc: 0.7560
Val Loss:   1.1700 | Val Acc:   0.5436

Epoch 4/30
-----------------------------------
Train Loss: 0.4558 | Train Acc: 0.7508
Val Loss:   1.1280 | Val Acc:   0.6114

Epoch 5/30
-----------------------------------
Train Loss: 0.4117 | Train Acc: 0.7667
Val Loss:   1.1602 | Val Acc:   0.5944

Epoch 6/30
-----------------------------------
Train Loss: 0.3307 | Train Acc: 0.8216
Val Loss:   1.3042 | Val Acc:   0.5702

Epoch 7/30
-----------------------------------
Train Loss: 0.3114 | Train Acc: 0.8259
Val Loss:   1.2662 | Val Acc:   0.5787

Epoch 8/30
-----------------------------------
Train 